# Laporan UTS Advance Computer Vision
**Klasifikasi Penyakit Daun Padi Menggunakan Convolutional Neural Network (CNN) dengan Arsitektur ResNet-50**

Nama: Wahyu Maulana  
NIM: 251012000008

In [2]:
from google.colab import drive
# Mencoba kembali menghubungkan Google Drive setelah restart runtime
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import os

# ==========================================
# 1. Konfigurasi Awal & Preprocessing Data
# ==========================================
DATASET_PATH = '/content/drive/MyDrive/Dataset_Daun_Padi/'
IMG_WIDTH, IMG_HEIGHT = 224, 224
BATCH_SIZE = 32
NUM_CLASSES = 4 # 3 Penyakit + 1 Sehat

# Validasi apakah folder dataset benar-benar ada di Drive
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Folder '{DATASET_PATH}' tidak ditemukan. "
        "Silakan unggah folder dataset Anda ke Google Drive terlebih dahulu "
        "atau pastikan penulisan nama foldernya sudah benar (perhatikan huruf besar/kecil)."
    )

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.15
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

FileNotFoundError: Folder '/content/drive/MyDrive/Dataset_Daun_Padi/' tidak ditemukan. Silakan unggah folder dataset Anda ke Google Drive terlebih dahulu atau pastikan penulisan nama foldernya sudah benar (perhatikan huruf besar/kecil).

In [ ]:
# ==========================================
# 2. Membangun Arsitektur Model (Transfer Learning)
# ==========================================
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_WIDTH, IMG_HEIGHT, 3))
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

# ==========================================
# 3. Kompilasi & Callbacks
# ==========================================
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
checkpoint = ModelCheckpoint('/content/best_resnet_model.h5', monitor='val_accuracy', save_best_only=True)

In [ ]:
# ==========================================
# 4. Pelatihan Model (Training)
# ==========================================
epochs = 30
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=epochs,
    callbacks=[early_stop, checkpoint]
)

# ==========================================
# 5. Visualisasi Hasil Pelatihan
# ==========================================
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Grafik Akurasi')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Grafik Loss')
plt.legend()
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# 6. Evaluasi Model (Classification Report & Confusion Matrix)
# ==========================================
test_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print("Memulai prediksi data evaluasi...")
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

class_names = list(test_generator.class_indices.keys())

print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Klasifikasi Penyakit Daun Padi')
plt.ylabel('Aktual (True Label)')
plt.xlabel('Prediksi (Predicted Label)')
plt.show()